# NQ Range XV 24 – Full Test: Barrier Strategy with Gradient Boosting

Same test battery as `train_barrier_NQ.ipynb`, but on **NQ Range XV 24 bars**: a new bar starts when the price has moved
a fixed range, so **price, not time, defines the chart**. Every horizon is therefore counted in **bars**, not minutes.

1. **Win models** (long / short): is `+target` reached before `−stop` within `horizon` bars? (timing / volatility)
2. **Pure direction test:** is the upper or the lower barrier touched first? (direction only, volatility removed)
3. **Trading simulations** with costs, one position at a time, chosen on validation and applied once to the test period,
   compared with *random direction*, *always long* and *always short* on the same entry bars.

**Bar-specific changes:** bars can form within the same second, so only exact duplicate rows are removed (never bars with
the same timestamp). The time a bar needed to form is a new input (market speed, from the timestamps only). TWAP is
weighted by that time. Session pauses are detected as gaps longer than `MAX_GAP_MINUTES`.

**One data file, no separate test CSV:** the last `TEST_MONTHS` months are the test period; of the data before it, the last
`VALIDATION_FRACTION` is validation, the rest training. The 1-minute results of NQ and GC are printed for comparison.

**Inputs:** only Open, Close, High, Low, Ask, Bid, Delta, Delta/Volume, Session Delta, Session Delta %, Max Delta,
Min Delta, Delta Change, Volume – plus values derived from the timestamp (time of day, bar duration), RSI / EMA / TWAP / VWAP
and the session structure. **Output:** the test period with all original columns plus probabilities and signals next to
the data file; models and log in `models_barrier_NQ_XV24/`.

## 0. Configuration

In [1]:
from pathlib import Path

SYMBOL = "NQ"
# File or folder; a missing ".csv" extension is added automatically. A folder with several CSV files is merged.
DATA_PATH = Path(r"E:\Programierung\Daten\NQXV24\Candelexport_NQ_24_RangeXV_08.08.2021_13.03.2026")
TEST_MONTHS = 12                # last 12 months of the file = test period (never used for training or selection)
MODEL_DIR = Path("models_barrier_NQ_XV24")
LOG_PATH = MODEL_DIR / "barrier_log.csv"

# Model inputs: ONLY these CSV columns (+ time features from "Time UTC" if USE_TIME_FEATURES)
INPUT_COLUMNS = [
    "Open", "Close", "High", "Low", "Ask", "Bid", "Delta", "Delta/Volume",
    "Session Delta", "Session Delta %", "Max Delta", "Min Delta", "Delta Change", "Volume",
]
PERCENT_COLUMNS = ["Delta/Volume", "Session Delta %"]   # exported as text "59.15%" -> 0.5915
USE_TIME_FEATURES = True                                 # hour of day + weekday from the timestamp

# Additional indicators (calculated only from the input columns above)
RSI_PERIODS = [7, 14, 21, 50]          # the set that worked best for GC
TWAP_WINDOWS = [15, 60]                # rolling TWAP over bars, weighted by bar duration (+ session TWAP)
VWAP_WINDOWS = [15, 60]                # rolling VWAP over bars (+ session VWAP, reset after every pause)
BAR_SPEED_WINDOWS = [5, 15, 60]        # rolling bar duration (how fast the bars form)
EMA_PERIODS = [9, 21, 50, 200]         # EMA over bars: distance of the close and slope over 5 bars
USE_SESSION_FEATURES = True            # previous session, night session, US opening range

# Session structure (New York time, daylight saving time handled automatically)
SESSION_START_HOUR_ET = 18      # CME Globex trading day: 18:00 -> 17:00 New York time (daily break 17:00-18:00)
US_OPEN_ET = (9, 30)            # US session open in New York time (= 13:30 UTC in summer, 14:30 UTC in winter)
OPENING_RANGE_MINUTES = 30      # opening range = first 30 minutes after the US open
# 1-minute results for comparison (direction AUC validation / test)
REFERENCE_RUNS = {
    "NQ 1-min, 15 min": (0.5141, 0.5117), "NQ 1-min, 60 min": (0.5233, 0.5067),
    "GC 1-min, 15 min": (0.5228, 0.5200), "GC 1-min, 60 min": (0.5152, 0.5165),
}

# Trade setups (horizon in BARS). Two kinds of barriers:
#   mode "vol":   target/stop = sigma x expected move over the horizon
#                 (std of bar-to-bar close changes over the last VOL_LOOKBACK bars x sqrt(horizon)), in ticks
#                 -> a trade means the same in quiet and in wild markets
#   mode "fixed": target_ticks / stop_ticks as fixed numbers, e.g.
#                 {"name": "10bars_fix", "horizon": 10, "mode": "fixed", "target_ticks": 48, "stop_ticks": 48}
SETUPS = [
    {"name": "10bars", "horizon": 10, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
    {"name": "30bars", "horizon": 30, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
]
VOL_LOOKBACK = 60               # candles for the volatility estimate (only past data)
MIN_BARRIER_TICKS = 100         # NQ Range XV 24: targets/stops never below 100 ticks

# Futures contract and execution
TICK_SIZE = 0.25                # NQ / MNQ: 0.25 index points
TICK_VALUE_USD = 5.0            # NQ: 5 USD per tick and contract (MNQ: 0.50 USD)
COST_PER_TRADE_TICKS = 2.0      # round trip incl. spread/slippage and commission -> adjust to your account
TARGET_FILL_EXTRA_TICKS = 1     # target counts only if the price trades this many ticks beyond it
MAX_GAP_MINUTES = 45            # range bars can take minutes to form -> only pauses > 45 min count as session break
WARMUP_CANDLES = 200            # first bars of a part are not traded (longest input: EMA 200)

# Data split and training
VALIDATION_FRACTION = 0.2       # last 20 % of the data before the test period -> threshold selection
TRAIN_ROW_STRIDE = 1            # 1 = every candle; e.g. 3 = every 3rd candle (faster, slightly less data)
GBM_PARAMS = dict(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=500,
    l2_regularization=1.0, random_state=42,
)

# Trading rule: trade if max(P_long, P_short) >= threshold; the threshold is chosen on validation (best t-value)
PROB_THRESHOLDS = [round(0.30 + 0.025 * i, 3) for i in range(21)]   # 0.30 ... 0.80
MIN_VALIDATION_TRADES = 100     # thresholds with fewer validation trades are not eligible
BASELINE_RANDOM_RUNS = 5
COMPUTE_FEATURE_IMPORTANCE = True   # permutation importance on a validation sample (a few minutes)

# Direction strategy (section 8.2): direction from the direction model, timing from the win models.
# All combinations are evaluated on validation; the one with the best t-value is applied ONCE to the test data.
DIRECTION_QUANTILES = [0.05, 0.10, 0.20]   # trade the lowest / highest X % of P(up first) -> short / long
TIMING_QUANTILES = [0.0, 0.5, 0.75]        # also require P(win) of the traded side >= this validation quantile (0 = off)
DIRECTION_SIDES = ["both", "short", "long"]

MODEL_DIR.mkdir(parents=True, exist_ok=True)


def resolve_data_files(path: Path) -> list[Path]:
    """File, file without .csv extension, or folder with CSV files."""
    if path.is_file():
        return [path]
    with_ext = Path(str(path) + ".csv")       # note: with_suffix() would cut at the ".2026" in the name
    if with_ext.is_file():
        return [with_ext]
    if path.is_dir():
        files = sorted(path.glob("*.csv"))
        if files:
            return files
    raise FileNotFoundError(f"No CSV found at {path} (also tried {with_ext} and *.csv inside a folder)")


DATA_FILES = resolve_data_files(DATA_PATH)
OUTPUT_CSV_PATH = DATA_FILES[0].with_name(DATA_FILES[0].stem + "_test_signals.csv")
print("Data files:", *[str(f) for f in DATA_FILES], sep="\n  ")
print(f"Output CSV: {OUTPUT_CSV_PATH}")
for s in SETUPS:
    if s.get("mode", "fixed") == "vol":
        print(f"Setup {s['name']}: target {s['target_sigma']} σ / stop {s['stop_sigma']} σ of the expected "
              f"move over {s['horizon']} bars (ticks are shown after the labels are computed)")
    else:
        be = (s["stop_ticks"] + COST_PER_TRADE_TICKS) / (s["target_ticks"] + s["stop_ticks"])
        print(f"Setup {s['name']}: +{s['target_ticks']} / -{s['stop_ticks']} ticks within {s['horizon']} bars "
              f"-> break-even win rate ≈ {100 * be:.1f} % (ignoring time exits)")

Data files:
  E:\Programierung\Daten\NQXV24\Candelexport_NQ_24_RangeXV_08.08.2021_13.03.2026.csv
Output CSV: E:\Programierung\Daten\NQXV24\Candelexport_NQ_24_RangeXV_08.08.2021_13.03.2026_test_signals.csv
Setup 10bars: target 1.0 σ / stop 1.0 σ of the expected move over 10 bars (ticks are shown after the labels are computed)
Setup 30bars: target 1.0 σ / stop 1.0 σ of the expected move over 30 bars (ticks are shown after the labels are computed)


## 1. Imports

In [2]:
import json
import time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

try:
    from IPython.display import display
except ImportError:  # outside Jupyter
    display = print

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

## 2. Load Data and Split off the Test Period

In [3]:
def load_csv(csv_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returns (data with input columns + timestamp + source row, original file as text)."""
    text = pd.read_csv(csv_path, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
    missing = [c for c in ["Time UTC"] + INPUT_COLUMNS if c not in text.columns]
    if missing:
        raise ValueError(f"{csv_path.name}: missing columns {missing}")
    df = pd.DataFrame({"_file": csv_path.name, "_row": np.arange(len(text))})
    for col in INPUT_COLUMNS:
        values = text[col].astype(str).str.strip()
        if col in PERCENT_COLUMNS:
            values = values.str.replace("%", "", regex=False).str.replace("'", "", regex=False).str.replace(" ", "", regex=False)
            values = values.where(values.str.contains(".", regex=False), values.str.replace(",", ".", regex=False))
            num = pd.to_numeric(values, errors="coerce") / 100
            n_bad = int((num.isna() & (values != "") & (values.str.lower() != "nan")).sum())
            if n_bad:
                print(f"⚠ {csv_path.name} {col}: {n_bad:,} values not parseable -> 0")
            df[col] = num.fillna(0.0)
        else:
            df[col] = pd.to_numeric(values, errors="coerce")
    ts = pd.to_datetime(text["Time UTC"], format="%d.%m.%Y %H:%M:%S", utc=True, errors="coerce")
    if ts.isna().mean() > 0.01:                    # other date format -> flexible parsing
        ts = pd.to_datetime(text["Time UTC"], dayfirst=True, utc=True, errors="coerce")
    df["timestamp"] = ts
    return df, text


parts, texts = [], {}
for f in DATA_FILES:
    d, t = load_csv(f)
    parts.append(d)
    texts[f.name] = t
    print(f"{f.name}: {len(d):,} rows")
df_full = pd.concat(parts, ignore_index=True)

# Clean: valid timestamps and OHLC, chronological order. Range bars can share a timestamp (several bars within one
# second), so only EXACT duplicate rows are removed (e.g. overlapping exports), never bars with the same time.
n0 = len(df_full)
df_full = df_full.dropna(subset=["timestamp", "Open", "High", "Low", "Close"])
df_full = df_full.sort_values(["timestamp", "_file", "_row"], kind="stable")
dupes = int(df_full.duplicated(subset=["timestamp"] + INPUT_COLUMNS, keep="last").sum())
df_full = df_full.drop_duplicates(subset=["timestamp"] + INPUT_COLUMNS, keep="last").reset_index(drop=True)
for col in INPUT_COLUMNS:
    df_full[col] = df_full[col].fillna(0.0).astype("float64")
print(f"Cleaned: {len(df_full):,} bars ({n0 - len(df_full) - dupes:,} invalid rows, {dupes:,} exact duplicates removed), "
      f"{df_full['timestamp'].min()} → {df_full['timestamp'].max()}")

# Test period = last TEST_MONTHS months
test_start = df_full["timestamp"].max() - pd.DateOffset(months=TEST_MONTHS)
n_before_test = int((df_full["timestamp"] < test_start).sum())
df_train = df_full.iloc[:n_before_test].reset_index(drop=True)
df_test = df_full.iloc[n_before_test:].reset_index(drop=True)
print(f"Training + validation: {df_train['timestamp'].min()} → {df_train['timestamp'].max()} ({len(df_train):,} bars)")
print(f"Test period:           {df_test['timestamp'].min()} → {df_test['timestamp'].max()} ({len(df_test):,} bars)")

Candelexport_NQ_24_RangeXV_08.08.2021_13.03.2026.csv: 632,478 rows
Cleaned: 632,478 bars (0 invalid rows, 0 exact duplicates removed), 2021-08-08 22:00:00+00:00 → 2026-03-13 13:19:06+00:00
Training + validation: 2021-08-08 22:00:00+00:00 → 2025-03-13 13:16:54+00:00 (408,374 bars)
Test period:           2025-03-13 13:19:08+00:00 → 2026-03-13 13:19:06+00:00 (224,104 bars)


## 3. Inputs (features)

Only backward-looking values of the input columns: candle shape, price changes, volatility and position in the recent range
(all in ticks, independent of the price level), plus order flow (Ask/Bid, Delta, Session Delta, Max/Min Delta, Volume)
as raw values, ratios and rolling sums. Gradient boosting handles missing values itself.

**Indicators:** RSI for every period in `RSI_PERIODS` (level + change over 5 minutes), **TWAP** and **VWAP** – each as
session value (reset at every session start / after a pause) and as rolling value over `TWAP_WINDOWS` / `VWAP_WINDOWS`.
TWAP/VWAP enter as **distance of the close in ticks** (and relative to the recent candle range), so the model does not
see absolute price levels. Typical price = (High + Low + Close) / 3.

**Session structure** (trading day = 18:00 → 17:00 New York time, like CME Globex):
* **Previous session:** high, low, close and VWAP → distance of the current close in ticks, position inside the previous
  range, previous range; Session Delta at the end of the previous session, its delta sum and delta per volume.
* **Night session (Asia/London):** from the session start until the US open – range so far, position inside it, distance
  to night high/low; after the US open the final night range is used. Flag whether the candle is in the night session.
* **US opening range:** high/low of the first `OPENING_RANGE_MINUTES` after the US open; available only **after** it is
  complete (no look-ahead): range, distance to OR high/low, +1 above / −1 below / 0 inside. Minutes since the US open.

All values use only completed data up to the current candle.

In [4]:
def rsi(close: pd.Series, period: int) -> pd.Series:
    """Relative Strength Index with Wilder smoothing (0..100)."""
    diff = close.diff()
    avg_gain = diff.clip(lower=0).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = (-diff.clip(upper=0)).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    value = 100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan))
    value[(avg_loss == 0) & (avg_gain > 0)] = 100.0
    value[(avg_loss == 0) & (avg_gain == 0)] = 50.0
    return value


def session_structure_features(df: pd.DataFrame) -> dict:
    """Previous session, night session and US opening range (New York time, DST-aware)."""
    t = TICK_SIZE
    c, h, l, vol = df["Close"], df["High"], df["Low"], df["Volume"]
    ts_et = df["timestamp"].dt.tz_convert("America/New_York")
    trade_day = (ts_et + pd.Timedelta(hours=24 - SESSION_START_HOUR_ET)).dt.date   # 18:00 ET -> next trading day
    tod = ts_et.dt.hour * 60 + ts_et.dt.minute
    us_open = US_OPEN_ET[0] * 60 + US_OPEN_ET[1]
    same_calendar_day = ts_et.dt.date == trade_day
    f = {}

    # ---- previous session (completed trading day before the current one)
    tp = (h + l + c) / 3
    g = pd.DataFrame({"day": trade_day, "h": h, "l": l, "c": c, "pv": tp * vol, "v": vol,
                      "sd": df["Session Delta"], "d": df["Delta"]})
    daily = g.groupby("day", sort=True).agg(high=("h", "max"), low=("l", "min"), close=("c", "last"),
                                            pv=("pv", "sum"), v=("v", "sum"), sd_end=("sd", "last"),
                                            delta_sum=("d", "sum"))
    daily["vwap"] = daily["pv"] / daily["v"].replace(0, np.nan)
    prev = daily.shift(1).reindex(trade_day.to_numpy())
    prev.index = df.index
    prev_range = (prev["high"] - prev["low"]).replace(0, np.nan)
    f["dist_prev_high_ticks"] = (c - prev["high"]) / t
    f["dist_prev_low_ticks"] = (c - prev["low"]) / t
    f["dist_prev_close_ticks"] = (c - prev["close"]) / t
    f["dist_prev_vwap_ticks"] = (c - prev["vwap"]) / t
    f["prev_range_ticks"] = prev_range / t
    f["pos_in_prev_range"] = (c - prev["low"]) / prev_range
    f["prev_session_delta"] = prev["sd_end"]
    f["prev_delta_sum"] = prev["delta_sum"]
    f["prev_delta_per_volume"] = prev["delta_sum"] / prev["v"].replace(0, np.nan)

    # ---- night session: session start until the US open (running high/low, fixed after the open)
    is_night = (~same_calendar_day) | (tod < us_open)
    night_high = h.where(is_night).groupby(trade_day).cummax().groupby(trade_day).ffill()
    night_low = l.where(is_night).groupby(trade_day).cummin().groupby(trade_day).ffill()
    night_range = (night_high - night_low).replace(0, np.nan)
    f["is_night_session"] = is_night.astype("float32")
    f["night_range_ticks"] = night_range / t
    f["pos_in_night_range"] = (c - night_low) / night_range
    f["dist_night_high_ticks"] = (c - night_high) / t
    f["dist_night_low_ticks"] = (c - night_low) / t

    # ---- US opening range: only after it is complete
    in_or = same_calendar_day & (tod >= us_open) & (tod < us_open + OPENING_RANGE_MINUTES)
    or_high = h.where(in_or).groupby(trade_day).transform("max")
    or_low = l.where(in_or).groupby(trade_day).transform("min")
    or_done = same_calendar_day & (tod >= us_open + OPENING_RANGE_MINUTES)
    or_high, or_low = or_high.where(or_done), or_low.where(or_done)
    f["or_range_ticks"] = (or_high - or_low) / t
    f["dist_or_high_ticks"] = (c - or_high) / t
    f["dist_or_low_ticks"] = (c - or_low) / t
    f["or_position"] = np.where(or_high.isna(), np.nan, np.where(c > or_high, 1.0, np.where(c < or_low, -1.0, 0.0)))
    f["minutes_since_us_open"] = np.where(same_calendar_day, tod - us_open, tod - us_open - 1440).astype("float32")
    return f


def session_ids(ts: pd.Series) -> pd.Series:
    """New session after every pause > MAX_GAP_MINUTES (daily break, weekend)."""
    return (ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES).cumsum()


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    o, h, l, c = df["Open"], df["High"], df["Low"], df["Close"]
    vol, ask, bid, delta = df["Volume"], df["Ask"], df["Bid"], df["Delta"]
    t = TICK_SIZE
    f = {}

    # Candle shape and price changes (ticks)
    f["body_ticks"] = (c - o) / t
    f["range_ticks"] = (h - l) / t
    f["upper_wick_ticks"] = (h - np.maximum(o, c)) / t
    f["lower_wick_ticks"] = (np.minimum(o, c) - l) / t
    ret1 = (c - c.shift(1)) / t
    for k in [1, 3, 5, 10, 15, 30, 60]:
        f[f"ret_{k}_ticks"] = (c - c.shift(k)) / t
    for k in [5, 15, 60]:
        hh, ll = h.rolling(k).max(), l.rolling(k).min()
        f[f"range_mean_{k}"] = f["range_ticks"].rolling(k).mean()
        f[f"ret_std_{k}"] = ret1.rolling(k).std()
        f[f"pos_in_range_{k}"] = (c - ll) / (hh - ll).replace(0, np.nan)
        f[f"dist_high_{k}_ticks"] = (hh - c) / t
        f[f"dist_low_{k}_ticks"] = (c - ll) / t

    # Order flow
    f["volume"] = vol
    f["log_volume"] = np.log1p(vol.clip(lower=0))
    f["ask"] = ask
    f["bid"] = bid
    f["ask_bid_imbalance"] = (ask - bid) / (ask + bid).replace(0, np.nan)
    f["delta"] = delta
    f["delta_per_volume"] = df["Delta/Volume"]
    f["session_delta"] = df["Session Delta"]
    f["session_delta_pct"] = df["Session Delta %"]
    f["session_delta_change_15"] = df["Session Delta"] - df["Session Delta"].shift(15)
    f["max_delta"] = df["Max Delta"]
    f["min_delta"] = df["Min Delta"]
    f["delta_change"] = df["Delta Change"]
    f["max_delta_rel"] = df["Max Delta"] / vol.replace(0, np.nan)
    f["min_delta_rel"] = df["Min Delta"] / vol.replace(0, np.nan)
    for k in [1, 2, 3, 5]:
        f[f"delta_lag_{k}"] = delta.shift(k)
        f[f"delta_per_volume_lag_{k}"] = df["Delta/Volume"].shift(k)
    for k in [5, 15, 60]:
        f[f"delta_sum_{k}"] = delta.rolling(k).sum()
        f[f"delta_ratio_{k}"] = delta.rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
        f[f"volume_rel_{k}"] = vol / vol.rolling(k).mean().replace(0, np.nan)
        f[f"ask_bid_imbalance_{k}"] = ((ask - bid).rolling(k).sum()
                                       / (ask + bid).rolling(k).sum().replace(0, np.nan))

    # RSI
    for p in RSI_PERIODS:
        r = rsi(c, p)
        f[f"rsi_{p}"] = r
        f[f"rsi_{p}_change_5"] = r - r.shift(5)

    # TWAP / VWAP (distance of the close in ticks and relative to the recent candle range)
    tp = (h + l + c) / 3
    session = session_ids(df["timestamp"])
    range_60 = ((h - l) / t).rolling(60, min_periods=5).mean().replace(0, np.nan)
    # Bar duration (seconds since the previous bar; backward-looking only). Pauses are not counted as duration.
    bar_sec = df["timestamp"].diff().dt.total_seconds()
    bar_sec = bar_sec.where(bar_sec <= MAX_GAP_MINUTES * 60)
    weight = bar_sec.fillna(bar_sec.median()).clip(lower=1.0)
    twap_session = (tp * weight).groupby(session).cumsum() / weight.groupby(session).cumsum()
    vol_cum = vol.groupby(session).cumsum()
    vwap_session = ((tp * vol).groupby(session).cumsum() / vol_cum.replace(0, np.nan)).groupby(session).ffill()
    averages = {"twap_session": twap_session, "vwap_session": vwap_session}
    for k in TWAP_WINDOWS:
        averages[f"twap_{k}"] = (tp * weight).rolling(k).sum() / weight.rolling(k).sum()
    for k in VWAP_WINDOWS:
        averages[f"vwap_{k}"] = (tp * vol).rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
    for name, avg in averages.items():
        f[f"dist_{name}_ticks"] = (c - avg) / t
        f[f"dist_{name}_rel"] = (c - avg) / t / range_60
    f["vwap_minus_twap_session_ticks"] = (vwap_session - twap_session) / t

    # EMA: distance of the close (ticks and relative to the recent bar range) and slope over 5 bars
    for p in EMA_PERIODS:
        ema = c.ewm(span=p, adjust=False, min_periods=p).mean()
        f[f"dist_ema_{p}_ticks"] = (c - ema) / t
        f[f"dist_ema_{p}_rel"] = (c - ema) / t / range_60
        f[f"ema_{p}_slope_5_ticks"] = (ema - ema.shift(5)) / t

    # Market speed: how long the bars take to form
    f["bar_seconds"] = bar_sec
    f["log_bar_seconds"] = np.log1p(bar_sec)
    for k in BAR_SPEED_WINDOWS:
        f[f"bar_seconds_mean_{k}"] = bar_sec.rolling(k, min_periods=1).mean()
        f[f"bar_speed_rel_{k}"] = bar_sec / bar_sec.rolling(k, min_periods=1).median().replace(0, np.nan)
    f["minutes_since_session_start"] = (df["timestamp"] - df["timestamp"].groupby(session).transform("first")).dt.total_seconds() / 60

    # Session structure over several days
    if USE_SESSION_FEATURES:
        f.update(session_structure_features(df))

    # Time of day / weekday from the timestamp (UTC)
    if USE_TIME_FEATURES:
        ts = df["timestamp"]
        minute_of_day = ts.dt.hour * 60 + ts.dt.minute
        f["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
        f["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)
        f["weekday"] = ts.dt.dayofweek
        f["bars_since_session_start"] = minutes_since_gap(ts)

    out = pd.DataFrame(f).replace([np.inf, -np.inf], np.nan).astype("float32")
    return out


def minutes_since_gap(ts: pd.Series) -> pd.Series:
    """Number of bars since the last pause > MAX_GAP_MINUTES (session start / after the weekend)."""
    gap = ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES
    block = gap.cumsum()
    return ts.groupby(block).cumcount().astype("float32")


t0 = time.time()
X_full = build_features(df_full)             # continuous file: no warm-up gap at the start of the test period
X_train_all = X_full.iloc[:n_before_test].reset_index(drop=True)
X_test = X_full.iloc[n_before_test:].reset_index(drop=True)
del X_full
FEATURE_COLUMNS = list(X_train_all.columns)
print(f"{len(FEATURE_COLUMNS)} inputs built in {time.time() - t0:.0f} s")
SESSION_PREFIXES = ("rsi_", "dist_twap", "dist_vwap", "vwap_minus", "dist_prev", "prev_", "pos_in_prev", "is_night",
                    "night_", "pos_in_night", "dist_night", "or_", "dist_or", "minutes_since_us_open", "bar_seconds", "log_bar_seconds",
                    "bar_speed", "minutes_since_session", "dist_ema", "ema_")
print("Indicator and session inputs:", [c for c in FEATURE_COLUMNS if c.startswith(SESSION_PREFIXES)])

126 inputs built in 2 s
Indicator and session inputs: ['rsi_7', 'rsi_7_change_5', 'rsi_14', 'rsi_14_change_5', 'rsi_21', 'rsi_21_change_5', 'rsi_50', 'rsi_50_change_5', 'dist_twap_session_ticks', 'dist_twap_session_rel', 'dist_vwap_session_ticks', 'dist_vwap_session_rel', 'dist_twap_15_ticks', 'dist_twap_15_rel', 'dist_twap_60_ticks', 'dist_twap_60_rel', 'dist_vwap_15_ticks', 'dist_vwap_15_rel', 'dist_vwap_60_ticks', 'dist_vwap_60_rel', 'vwap_minus_twap_session_ticks', 'dist_ema_9_ticks', 'dist_ema_9_rel', 'ema_9_slope_5_ticks', 'dist_ema_21_ticks', 'dist_ema_21_rel', 'ema_21_slope_5_ticks', 'dist_ema_50_ticks', 'dist_ema_50_rel', 'ema_50_slope_5_ticks', 'dist_ema_200_ticks', 'dist_ema_200_rel', 'ema_200_slope_5_ticks', 'bar_seconds', 'log_bar_seconds', 'bar_seconds_mean_5', 'bar_speed_rel_5', 'bar_seconds_mean_15', 'bar_speed_rel_15', 'bar_seconds_mean_60', 'bar_speed_rel_60', 'minutes_since_session_start', 'dist_prev_high_ticks', 'dist_prev_low_ticks', 'dist_prev_close_ticks', 'dist_

## 4. Labels: Barrier Outcomes for Every Candle

For every bar, a long and a short trade entered at its close are played through bar by bar:
target (+`TARGET_FILL_EXTRA_TICKS` beyond it) → win, stop → loss (stop wins if both are touched in the same candle),
neither → closed at the close after `horizon` bars. Windows that contain a pause are invalid (not traded, not trained).

With `mode: "vol"` target and stop are set per candle from the volatility of the **past** `VOL_LOOKBACK` candles.

**Direction label (for the pure direction test):** is the upper barrier (+target) touched **before** the lower one
(−target)? Only windows where exactly one side is reached first count; time-outs and candles touching both are excluded.
This removes the volatility question – what remains is only "up or down".

In [5]:
EXIT_TARGET, EXIT_STOP, EXIT_TIME = 1, 2, 3


def barrier_ticks(df: pd.DataFrame, setup: dict):
    """Target and stop in ticks per candle (arrays)."""
    n = len(df)
    if setup.get("mode", "fixed") == "fixed":
        return np.full(n, float(setup["target_ticks"])), np.full(n, float(setup["stop_ticks"]))
    ret1 = df["Close"].diff() / TICK_SIZE
    expected_move = (ret1.rolling(VOL_LOOKBACK, min_periods=VOL_LOOKBACK // 2).std() * np.sqrt(setup["horizon"])).to_numpy()
    target = np.maximum(np.round(setup["target_sigma"] * expected_move), MIN_BARRIER_TICKS)
    stop = np.maximum(np.round(setup["stop_sigma"] * expected_move), MIN_BARRIER_TICKS)
    return target, stop


def barrier_outcomes(df: pd.DataFrame, setup: dict) -> dict:
    horizon = setup["horizon"]
    target_ticks, stop_ticks = barrier_ticks(df, setup)
    c = df["Close"].to_numpy("float64")
    h = df["High"].to_numpy("float64")
    l = df["Low"].to_numpy("float64")
    n = len(df)
    extra = TARGET_FILL_EXTRA_TICKS * TICK_SIZE
    out = {"target_ticks": target_ticks, "stop_ticks": stop_ticks}
    for side, d in (("long", 1), ("short", -1)):
        target = c + d * (target_ticks * TICK_SIZE + extra)
        stop = c - d * stop_ticks * TICK_SIZE
        result = np.full(n, np.nan)
        exit_type = np.zeros(n, dtype="int8")
        minutes = np.zeros(n, dtype="int16")
        done = np.zeros(n, dtype=bool)
        for k in range(1, horizon + 1):
            hk = np.full(n, np.nan)
            lk = np.full(n, np.nan)
            hk[:n - k], lk[:n - k] = h[k:], l[k:]
            if d == 1:
                stop_hit, tgt_hit = lk <= stop, hk >= target
            else:
                stop_hit, tgt_hit = hk >= stop, lk <= target
            stop_hit &= ~done
            tgt_hit &= ~done & ~stop_hit
            result[stop_hit], exit_type[stop_hit], minutes[stop_hit] = -stop_ticks[stop_hit], EXIT_STOP, k
            result[tgt_hit], exit_type[tgt_hit], minutes[tgt_hit] = target_ticks[tgt_hit], EXIT_TARGET, k
            done |= stop_hit | tgt_hit
        ck = np.full(n, np.nan)
        ck[:n - horizon] = c[horizon:]
        open_ = ~done
        result[open_] = d * (ck[open_] - c[open_]) / TICK_SIZE
        exit_type[open_], minutes[open_] = EXIT_TIME, horizon
        out[side] = {"result": result, "exit": exit_type, "minutes": minutes, "win": exit_type == EXIT_TARGET}

    # Direction label: +target before -target (symmetric, no extra tick); both in one candle / time-out -> unresolved
    up_level, down_level = c + target_ticks * TICK_SIZE, c - target_ticks * TICK_SIZE
    up_first = np.zeros(n, dtype=bool)
    resolved = np.zeros(n, dtype=bool)
    decided = np.zeros(n, dtype=bool)
    for k in range(1, horizon + 1):
        hk = np.full(n, np.nan)
        lk = np.full(n, np.nan)
        hk[:n - k], lk[:n - k] = h[k:], l[k:]
        up, down = (hk >= up_level) & ~decided, (lk <= down_level) & ~decided
        only_up, only_down = up & ~down, down & ~up
        up_first |= only_up
        resolved |= only_up | only_down
        decided |= up | down
    out["direction"] = {"up_first": up_first, "resolved": resolved}

    # Valid = complete window inside the file, no pause inside the window, warm-up done
    gap = (df["timestamp"].diff().dt.total_seconds().div(60).fillna(0).to_numpy() > MAX_GAP_MINUTES).astype(int)
    cum = np.cumsum(gap)
    idx = np.arange(n)
    end = np.minimum(idx + horizon, n - 1)
    out["valid"] = ((idx + horizon < n) & ((cum[end] - cum[idx]) == 0) & (idx >= WARMUP_CANDLES)
                    & np.isfinite(target_ticks) & np.isfinite(stop_ticks))
    return out


t0 = time.time()
outcomes_train = {s["name"]: barrier_outcomes(df_train, s) for s in SETUPS}
outcomes_test = {s["name"]: barrier_outcomes(df_test, s) for s in SETUPS}
print(f"Barrier outcomes computed in {time.time() - t0:.0f} s")
for s in SETUPS:
    o = outcomes_train[s["name"]]
    v = o["valid"]
    print(f"[{s['name']}] valid candles {v.sum():,} | win rate if ALWAYS traded: "
          f"long {100 * o['long']['win'][v].mean():.1f} %, short {100 * o['short']['win'][v].mean():.1f} % "
          f"| Ø result long {np.nanmean(o['long']['result'][v]):+.2f}, short {np.nanmean(o['short']['result'][v]):+.2f} ticks (gross)")
    for label, oo in (("training data", o), ("test data", outcomes_test[s["name"]])):
        vv = oo["valid"]
        print(f"      {label}: Ø target {np.mean(oo['target_ticks'][vv]):.1f} / Ø stop {np.mean(oo['stop_ticks'][vv]):.1f} ticks, "
              f"direction resolved in {100 * oo['direction']['resolved'][vv].mean():.1f} % of the windows, "
              f"up first {100 * oo['direction']['up_first'][vv & oo['direction']['resolved']].mean():.1f} %")

Barrier outcomes computed in 1 s
[10bars] valid candles 382,168 | win rate if ALWAYS traded: long 32.7 %, short 33.1 % | Ø result long -0.87, short +0.21 ticks (gross)
      training data: Ø target 120.7 / Ø stop 120.7 ticks, direction resolved in 66.7 % of the windows, up first 49.7 %
      test data: Ø target 123.9 / Ø stop 123.9 ticks, direction resolved in 66.0 % of the windows, up first 50.1 %
[30bars] valid candles 353,993 | win rate if ALWAYS traded: long 32.3 %, short 33.3 % | Ø result long -2.31, short +1.63 ticks (gross)
      training data: Ø target 208.9 / Ø stop 208.9 ticks, direction resolved in 66.1 % of the windows, up first 49.3 %
      test data: Ø target 214.5 / Ø stop 214.5 ticks, direction resolved in 65.3 % of the windows, up first 50.0 %


## 5. Time Split: Training / Validation

In [6]:
n_train_total = len(df_train)
val_start = int(n_train_total * (1 - VALIDATION_FRACTION))
max_horizon = max(s["horizon"] for s in SETUPS)
# Purge: training labels must not look into the validation period
train_rows = np.arange(0, val_start - max_horizon, TRAIN_ROW_STRIDE)
val_rows = np.arange(val_start, n_train_total)
print(f"Training:   {df_train['timestamp'].iloc[train_rows[0]]} → {df_train['timestamp'].iloc[train_rows[-1]]} "
      f"({len(train_rows):,} bars, stride {TRAIN_ROW_STRIDE})")
print(f"Validation: {df_train['timestamp'].iloc[val_rows[0]]} → {df_train['timestamp'].iloc[val_rows[-1]]} ({len(val_rows):,} bars)")
print(f"Test:       {df_test['timestamp'].iloc[0]} → {df_test['timestamp'].iloc[-1]} ({len(df_test):,} bars)")

Training:   2021-08-08 22:00:00+00:00 → 2024-09-06 05:16:31+00:00 (326,669 bars, stride 1)
Validation: 2024-09-06 07:00:09+00:00 → 2025-03-13 13:16:54+00:00 (81,675 bars)
Test:       2025-03-13 13:19:08+00:00 → 2026-03-13 13:19:06+00:00 (224,104 bars)


## 6. Train Long and Short Classifier per Setup

In [7]:
models = {}
auc_rows = []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    tr = train_rows[o_tr["valid"][train_rows]]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    models[name] = {}
    for side in ("long", "short"):
        t0 = time.time()
        clf = HistGradientBoostingClassifier(**GBM_PARAMS)
        clf.fit(X_train_all.iloc[tr], o_tr[side]["win"][tr])
        p_val = clf.predict_proba(X_train_all.iloc[va])[:, 1]
        p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
        models[name][side] = clf
        auc_rows.append({
            "Setup": name, "Side": side, "Train candles": len(tr),
            "Base win rate val %": 100 * o_tr[side]["win"][va].mean(),
            "AUC validation": roc_auc_score(o_tr[side]["win"][va], p_val),
            "AUC test": roc_auc_score(o_te[side]["win"][te], p_te),
            "Ø P val": p_val.mean(), "P val 99%": np.percentile(p_val, 99),
            "Fit seconds": time.time() - t0,
        })
        print(f"[{name} {side}] trained in {time.time() - t0:.0f} s – AUC validation {auc_rows[-1]['AUC validation']:.4f}, "
              f"test {auc_rows[-1]['AUC test']:.4f}")

print("\nAUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful")
display(pd.DataFrame(auc_rows).round(4))

[10bars long] trained in 7 s – AUC validation 0.5515, test 0.5458
[10bars short] trained in 5 s – AUC validation 0.5467, test 0.5389
[30bars long] trained in 5 s – AUC validation 0.5315, test 0.5299
[30bars short] trained in 5 s – AUC validation 0.5280, test 0.5248

AUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful


,Setup,Side,Train candles,Base win rate val %,AUC validation,AUC test,Ø P val,P val 99%,Fit seconds
0,10bars,long,303826,32.5965,0.5515,0.5458,0.3282,0.4960,6.6331
1,10bars,short,303826,32.8289,0.5467,0.5389,0.3261,0.4846,5.0528
2,30bars,long,280178,31.8588,0.5315,0.5299,0.3323,0.6086,4.5707
3,30bars,short,280178,33.2805,0.5280,0.5248,0.3251,0.5602,4.5024


### 6.1 Which Inputs Matter (permutation importance on a validation sample)

In [8]:
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(0)
    for s in SETUPS:
        name = s["name"]
        va = val_rows[outcomes_train[name]["valid"][val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        for side in ("long", "short"):
            imp = permutation_importance(
                models[name][side], X_train_all.iloc[sample], outcomes_train[name][side]["win"][sample],
                scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
            )
            top = (pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False).head(12))
            print(f"[{name} {side}] AUC loss when an input is shuffled (top 12):")
            print("   " + ", ".join(f"{k} {v:.4f}" for k, v in top.items()))

[10bars long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0391, dist_high_60_ticks 0.0040, range_mean_60 0.0026, dist_prev_close_ticks 0.0018, delta_ratio_60 0.0017, prev_range_ticks 0.0014, dist_prev_high_ticks 0.0009, bars_since_session_start 0.0008, ret_60_ticks 0.0006, tod_cos 0.0006, session_delta 0.0006, bar_seconds_mean_5 0.0005
[10bars short] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0422, dist_night_low_ticks 0.0045, dist_prev_close_ticks 0.0027, bars_since_session_start 0.0020, dist_low_60_ticks 0.0016, bar_seconds_mean_60 0.0012, session_delta_pct 0.0012, bar_seconds_mean_15 0.0010, minutes_since_us_open 0.0009, ema_200_slope_5_ticks 0.0009, dist_prev_high_ticks 0.0009, dist_vwap_session_rel 0.0008
[30bars long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0254, dist_low_60_ticks 0.0027, or_range_ticks 0.0027, range_mean_60 0.0027, session_delta 0.0024, dist_twap_session_ticks 0.0023, dist_night_low_ticks 0.0020, dist_prev_

### 6.2 Pure Direction Test: "Up or Down First?"

One classifier per setup on the direction label only (volatility removed). **AUC 0.50 = no direction information**,
0.52–0.55 = weak, above 0.55 = interesting. The bucket table shows on the test data whether higher model probabilities
really go along with more "up first" outcomes – that is what a trading signal needs.

In [9]:
direction_models, direction_rows = {}, []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    ok_tr = o_tr["valid"] & o_tr["direction"]["resolved"]
    ok_te = o_te["valid"] & o_te["direction"]["resolved"]
    tr = train_rows[ok_tr[train_rows]]
    va = val_rows[ok_tr[val_rows]]
    te = np.flatnonzero(ok_te)
    t0 = time.time()
    clf = HistGradientBoostingClassifier(**GBM_PARAMS)
    clf.fit(X_train_all.iloc[tr], o_tr["direction"]["up_first"][tr])
    direction_models[name] = clf
    p_va = clf.predict_proba(X_train_all.iloc[va])[:, 1]
    p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
    y_va, y_te = o_tr["direction"]["up_first"][va], o_te["direction"]["up_first"][te]
    direction_rows.append({
        "Setup": name, "Train windows": len(tr), "Up first val %": 100 * y_va.mean(), "Up first test %": 100 * y_te.mean(),
        "AUC direction validation": roc_auc_score(y_va, p_va), "AUC direction test": roc_auc_score(y_te, p_te),
        "Fit seconds": time.time() - t0,
    })
    buckets = pd.qcut(p_te, 10, labels=False, duplicates="drop")
    table = (pd.DataFrame({"bucket": buckets, "p": p_te, "up": y_te})
             .groupby("bucket").agg(windows=("up", "size"), mean_p=("p", "mean"), actual_up_pct=("up", "mean")))
    table["actual_up_pct"] *= 100
    print(f"[{name}] test: model probability (10 buckets, low -> high) vs. actual share 'up first'")
    display(table.round(3))

print("AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting")
direction_df = pd.DataFrame(direction_rows)
display(direction_df.round(4))

# Comparison with the 1-minute runs (horizons differ: bars here, minutes there)
for _, r in direction_df.iterrows():
    print(f"[NQ Range XV 24, {r['Setup']}] direction AUC validation {r['AUC direction validation']:.4f} / "
          f"test {r['AUC direction test']:.4f}")
for label, (v, t_) in REFERENCE_RUNS.items():
    print(f"[{label}] direction AUC validation {v:.4f} / test {t_:.4f}")

# Which inputs does the DIRECTION model use? (permutation importance on resolved validation windows)
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(1)
    indicator_prefixes = SESSION_PREFIXES
    for s in SETUPS:
        name = s["name"]
        o_tr = outcomes_train[name]
        ok = o_tr["valid"] & o_tr["direction"]["resolved"]
        va = val_rows[ok[val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        imp = permutation_importance(
            direction_models[name], X_train_all.iloc[sample], o_tr["direction"]["up_first"][sample],
            scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
        )
        importance = pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False)
        print(f"\n[{name} DIRECTION] top 15 inputs (AUC loss when shuffled):")
        print("   " + ", ".join(f"{k} {v:.4f}" for k, v in importance.head(15).items()))
        ind = importance[[c for c in importance.index if c.startswith(indicator_prefixes)]]
        print(f"[{name} DIRECTION] all indicator and session inputs, best first:")
        display(ind.round(5).to_frame("AUC loss").T)

[10bars] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,14359,0.364,49.530
1,14358,0.435,50.049
2,14358,0.463,50.522
3,14358,0.482,49.652
4,14358,0.499,49.958
5,14358,0.515,50.460
6,14358,0.532,49.506
7,14358,0.553,50.996
8,14358,0.585,50.501


[30bars] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,13635,0.289,50.730
1,13635,0.389,49.358
2,13635,0.434,48.720
3,13634,0.467,49.435
4,13635,0.496,49.754
5,13635,0.523,50.554
6,13634,0.552,49.787
7,13635,0.586,50.722
8,13635,0.636,50.634


AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting


,Setup,Train windows,Up first val %,Up first test %,AUC direction validation,AUC direction test,Fit seconds
0,10bars,202969,49.8062,50.1086,0.5011,0.5016,2.8946
1,30bars,185607,48.8981,49.9758,0.4993,0.5034,2.7099


[NQ Range XV 24, 10bars] direction AUC validation 0.5011 / test 0.5016
[NQ Range XV 24, 30bars] direction AUC validation 0.4993 / test 0.5034
[NQ 1-min, 15 min] direction AUC validation 0.5141 / test 0.5117
[NQ 1-min, 60 min] direction AUC validation 0.5233 / test 0.5067
[GC 1-min, 15 min] direction AUC validation 0.5228 / test 0.5200
[GC 1-min, 60 min] direction AUC validation 0.5152 / test 0.5165

[10bars DIRECTION] top 15 inputs (AUC loss when shuffled):
   dist_prev_close_ticks 0.0034, bars_since_session_start 0.0034, minutes_since_session_start 0.0027, prev_range_ticks 0.0020, delta_ratio_60 0.0019, dist_low_60_ticks 0.0016, session_delta_pct 0.0012, dist_twap_session_ticks 0.0010, dist_prev_high_ticks 0.0008, dist_or_low_ticks 0.0006, dist_ema_200_rel 0.0005, pos_in_night_range 0.0005, dist_night_low_ticks 0.0004, dist_prev_vwap_ticks 0.0004, dist_vwap_session_ticks 0.0003
[10bars DIRECTION] all indicator and session inputs, best first:


,dist_prev_close_ticks,minutes_since_session_start,prev_range_ticks,dist_twap_session_ticks,dist_prev_high_ticks,dist_or_low_ticks,dist_ema_200_rel,pos_in_night_range,dist_night_low_ticks,dist_prev_vwap_ticks,dist_vwap_session_ticks,dist_ema_200_ticks,ema_50_slope_5_ticks,ema_9_slope_5_ticks,dist_vwap_session_rel,bar_seconds_mean_5,dist_ema_50_rel,dist_twap_60_rel,bar_speed_rel_15,rsi_7,dist_vwap_15_ticks,rsi_21_change_5,rsi_7_change_5,dist_ema_50_ticks,dist_ema_21_ticks,...,rsi_14,dist_ema_9_rel,is_night_session,bar_speed_rel_5,dist_vwap_15_rel,dist_ema_21_rel,dist_vwap_60_ticks,bar_seconds,dist_vwap_60_rel,or_range_ticks,bar_seconds_mean_15,dist_night_high_ticks,dist_twap_15_ticks,rsi_50_change_5,bar_seconds_mean_60,ema_21_slope_5_ticks,minutes_since_us_open,dist_twap_session_rel,dist_or_high_ticks,dist_prev_low_ticks,night_range_ticks,vwap_minus_twap_session_ticks,pos_in_prev_range,prev_session_delta,prev_delta_per_volume
AUC loss,0.00345,0.0027,0.002,0.00103,0.00077,0.00062,0.00054,0.00049,0.00039,0.00038,0.00034,0.00032,0.00023,0.00021,0.00017,0.00015,0.00014,0.00014,0.00013,0.00012,0.00011,0.0001,0.00009,0.00007,0.00006,...,-0.00004,-0.00004,-0.00004,-0.00004,-0.00005,-0.00005,-0.00005,-0.00008,-0.00016,-0.00017,-0.00028,-0.00031,-0.00032,-0.00033,-0.00048,-0.00055,-0.00067,-0.00077,-0.00095,-0.00123,-0.00159,-0.00163,-0.00189,-0.0019,-0.00393



[30bars DIRECTION] top 15 inputs (AUC loss when shuffled):
   or_range_ticks 0.0051, delta_ratio_60 0.0038, night_range_ticks 0.0035, bars_since_session_start 0.0030, tod_cos 0.0027, dist_prev_high_ticks 0.0025, dist_prev_close_ticks 0.0023, session_delta 0.0017, minutes_since_session_start 0.0017, ret_60_ticks 0.0016, dist_prev_vwap_ticks 0.0016, dist_night_low_ticks 0.0015, bar_seconds_mean_15 0.0013, dist_night_high_ticks 0.0013, ret_std_60 0.0012
[30bars DIRECTION] all indicator and session inputs, best first:


,or_range_ticks,night_range_ticks,dist_prev_high_ticks,dist_prev_close_ticks,minutes_since_session_start,dist_prev_vwap_ticks,dist_night_low_ticks,bar_seconds_mean_15,dist_night_high_ticks,pos_in_prev_range,minutes_since_us_open,prev_range_ticks,pos_in_night_range,dist_or_high_ticks,dist_twap_60_ticks,bar_speed_rel_15,dist_twap_60_rel,rsi_50,bar_seconds,ema_9_slope_5_ticks,rsi_14,dist_ema_9_rel,dist_twap_15_rel,rsi_21,dist_vwap_15_rel,...,rsi_14_change_5,rsi_50_change_5,dist_ema_50_rel,dist_vwap_60_rel,dist_twap_15_ticks,dist_ema_50_ticks,dist_vwap_session_rel,dist_vwap_15_ticks,ema_50_slope_5_ticks,is_night_session,dist_vwap_60_ticks,rsi_7,bar_seconds_mean_5,ema_21_slope_5_ticks,dist_ema_200_ticks,dist_or_low_ticks,vwap_minus_twap_session_ticks,dist_twap_session_ticks,ema_200_slope_5_ticks,dist_twap_session_rel,dist_ema_200_rel,dist_prev_low_ticks,bar_seconds_mean_60,prev_session_delta,prev_delta_per_volume
AUC loss,0.00512,0.00352,0.00245,0.00229,0.00172,0.00158,0.00148,0.00132,0.00131,0.00121,0.0009,0.00069,0.00069,0.00053,0.00041,0.00033,0.00028,0.00026,0.00008,0.00007,0.00004,0.00003,0.00003,0.00002,0.00002,...,-0.00002,-0.00003,-0.00004,-0.00004,-0.00008,-0.00008,-0.00009,-0.00011,-0.00011,-0.00012,-0.00014,-0.00014,-0.00017,-0.00018,-0.00044,-0.0006,-0.00071,-0.00081,-0.00105,-0.00172,-0.00172,-0.0019,-0.00246,-0.00284,-0.00311


## 7. Trading Simulation and Threshold Selection (validation only)

One position at a time. At every candle: `P_long`, `P_short`; if the larger one is ≥ threshold, a trade in that direction is
opened at the close; its result and duration come from the barrier outcome of that candle. Net = gross − costs.
The threshold with the best **t-value** (Ø net ÷ std × √trades) and at least `MIN_VALIDATION_TRADES` trades is selected.

In [10]:
def simulate(outcome: dict, rows: np.ndarray, direction: np.ndarray) -> pd.DataFrame:
    """rows: candidate candles in time order, direction: +1 / -1 per candidate."""
    trades, next_free = [], -1
    for r, d in zip(rows, direction):
        if r < next_free:
            continue
        side = outcome["long"] if d == 1 else outcome["short"]
        trades.append({"t": int(r), "direction": int(d), "gross_ticks": float(side["result"][r]),
                       "exit": int(side["exit"][r]), "minutes": int(side["minutes"][r])})
        next_free = r + int(side["minutes"][r])
    return pd.DataFrame(trades)


def summarize(trades: pd.DataFrame, timestamps: pd.Series, days: int) -> dict:
    """timestamps: all candles of the data set (trade rows index into it), days: trading days of the period."""
    if trades.empty:
        return {"Trades": 0}
    net = trades["gross_ticks"].to_numpy() - COST_PER_TRADE_TICKS
    equity = np.concatenate([[0.0], np.cumsum(net * TICK_VALUE_USD)])
    drawdown = (np.maximum.accumulate(equity) - equity).max()
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    ts = timestamps.iloc[trades["t"].to_numpy()]
    monthly = pd.Series(net, index=ts.dt.strftime("%Y-%m").to_numpy()).groupby(level=0).sum()
    days = max(days, 1)
    is_long = trades["direction"].to_numpy() == 1
    return {
        "Trades": len(trades), "Trades/day": len(trades) / days, "Long %": 100 * is_long.mean(),
        "Target %": 100 * (trades["exit"] == EXIT_TARGET).mean(),
        "Stop %": 100 * (trades["exit"] == EXIT_STOP).mean(),
        "Time exit %": 100 * (trades["exit"] == EXIT_TIME).mean(),
        "Ø net (ticks)": net.mean(),
        "Ø net long": net[is_long].mean() if is_long.any() else np.nan,
        "Ø net short": net[~is_long].mean() if (~is_long).any() else np.nan,
        "Profit factor": wins / losses if losses > 0 else np.inf,
        "t-value": net.mean() / net.std(ddof=1) * np.sqrt(len(net)) if len(net) > 1 and net.std(ddof=1) > 0 else np.nan,
        "Months in profit": f"{int((monthly > 0).sum())}/{len(monthly)}",
        "Net total USD/contract": net.sum() * TICK_VALUE_USD,
        "Max drawdown USD": drawdown,
    }


def signals(p_long: np.ndarray, p_short: np.ndarray, threshold: float):
    best = np.maximum(p_long, p_short)
    mask = best >= threshold
    direction = np.where(p_long >= p_short, 1, -1)
    return mask, direction


probs = {"val": {}, "test": {}}
chosen = {}
for s in SETUPS:
    name = s["name"]
    o_tr = outcomes_train[name]
    va = val_rows[o_tr["valid"][val_rows]]
    p_l = models[name]["long"].predict_proba(X_train_all.iloc[va])[:, 1]
    p_s = models[name]["short"].predict_proba(X_train_all.iloc[va])[:, 1]
    probs["val"][name] = (va, p_l, p_s)

    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(o_tr, va[mask], direction[mask])
        rows.append({"Threshold": thr, "Signal candles": int(mask.sum()),
                     **summarize(tr, df_train["timestamp"], val_days)})
    val_table = pd.DataFrame(rows)
    eligible = val_table[val_table["Trades"] >= MIN_VALIDATION_TRADES]
    if eligible.empty or eligible["t-value"].isna().all():
        chosen[name] = None
        print(f"[{name}] no threshold with ≥ {MIN_VALIDATION_TRADES} validation trades – setup not tradable")
    else:
        best = eligible.loc[eligible["t-value"].idxmax()]
        chosen[name] = float(best["Threshold"])
        print(f"[{name}] chosen threshold {chosen[name]:.3f}: validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
              f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")
    print(f"[{name}] validation results per threshold:")
    display(val_table.round(3))

[10bars] chosen threshold 0.550: validation Ø net +14.47 ticks, t-value 1.53, 119 trades
[10bars] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,64862,9658,60.362,51.439,34.138,34.645,31.218,-2.449,-2.745,-2.135,0.951,-2.330,2/7,-118250.0,136420.0
1,0.325,56510,8851,55.319,51.734,35.250,35.058,29.692,-1.888,-2.223,-1.529,0.962,-1.728,1/7,-83565.0,100515.0
2,0.350,45092,7655,47.844,52.567,35.833,35.258,28.909,-1.323,-1.894,-0.690,0.973,-1.133,3/7,-50620.0,96385.0
3,0.375,30892,6003,37.519,52.824,35.649,36.698,27.653,-2.851,-2.304,-3.464,0.942,-2.172,1/7,-85575.0,113490.0
4,0.400,17945,4031,25.194,53.659,36.765,37.112,26.123,-2.157,-3.821,-0.231,0.956,-1.356,2/7,-43475.0,61010.0
5,0.425,9510,2413,15.081,53.626,36.925,38.375,24.700,-3.091,-4.087,-1.938,0.938,-1.502,3/7,-37290.0,62720.0
6,0.450,4699,1305,8.156,53.180,39.004,38.238,22.759,0.031,0.974,-1.039,1.001,0.011,2/7,205.0,35740.0
7,0.475,2351,711,4.444,54.571,40.084,38.678,21.238,-1.114,-3.570,1.836,0.977,-0.292,2/7,-3960.0,21085.0
8,0.500,1226,369,2.306,55.285,42.547,36.585,20.867,4.864,-0.270,11.212,1.107,0.930,5/7,8975.0,11635.0
9,0.525,657,185,1.156,54.595,44.865,37.297,17.838,8.200,-0.386,18.524,1.184,1.093,3/7,7585.0,7220.0


[30bars] chosen threshold 0.375: validation Ø net +0.87 ticks, t-value 0.26, 2755 trades
[30bars] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,63525,3473,23.153,50.965,33.832,33.487,32.681,-2.614,-4.656,-0.491,0.968,-0.873,2/7,-45390.0,105160.0
1,0.325,56948,3329,22.193,51.307,34.335,33.253,32.412,-0.406,-2.741,2.055,0.995,-0.133,4/7,-6750.0,68005.0
2,0.350,48595,3071,20.473,51.840,34.354,33.116,32.530,-0.948,-4.316,2.678,0.988,-0.300,3/7,-14550.0,102070.0
3,0.375,39076,2755,18.367,51.906,34.918,33.358,31.724,0.872,-1.746,3.697,1.011,0.263,5/7,12005.0,69605.0
4,0.400,29668,2334,15.560,52.999,35.518,34.404,30.077,-0.136,-4.785,5.107,0.998,-0.038,4/7,-1585.0,48630.0
5,0.425,21510,1884,12.560,53.662,35.934,35.244,28.822,-2.864,-6.772,1.661,0.965,-0.716,3/7,-26980.0,58290.0
6,0.450,14786,1450,9.667,54.483,37.103,34.759,28.138,-1.021,-4.051,2.606,0.988,-0.223,5/7,-7400.0,51710.0
7,0.475,9742,1019,6.793,55.741,33.955,35.525,30.520,-6.081,-9.901,-1.271,0.927,-1.131,3/7,-30985.0,61960.0
8,0.500,6257,732,4.880,56.967,35.383,33.880,30.738,-1.734,-10.468,9.829,0.979,-0.273,3/7,-6345.0,44645.0
9,0.525,3960,481,3.207,58.212,36.798,35.343,27.859,-1.869,-4.579,1.905,0.977,-0.239,3/7,-4495.0,36610.0


## 8. Test: Model vs. Random Direction vs. Always Long

Applied **once** with the threshold chosen on validation. The comparisons use exactly the same entry candles.

In [11]:
test_days = df_test["timestamp"].dt.date.nunique()
test_rows = []
test_trades = {}
for s in SETUPS:
    name = s["name"]
    o_te = outcomes_test[name]
    te = np.flatnonzero(o_te["valid"])
    p_l = models[name]["long"].predict_proba(X_test.iloc[te])[:, 1]
    p_s = models[name]["short"].predict_proba(X_test.iloc[te])[:, 1]
    probs["test"][name] = (te, p_l, p_s)
    thr = chosen[name]
    if thr is None:
        continue
    mask, direction = signals(p_l, p_s, thr)
    rows_sig = te[mask]
    strategies = {
        "Model": [direction[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=mask.sum()) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(mask.sum(), dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for d in dirs:
            tr = simulate(o_te, rows_sig, d)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Model":
                test_trades[name] = tr
        row = {"Setup": name, "Threshold": thr, "Strategy": strategy, "Signal candles": int(mask.sum())}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        test_rows.append(row)

test_df = pd.DataFrame(test_rows)
print(f"Rules: costs {COST_PER_TRADE_TICKS} ticks, target fill +{TARGET_FILL_EXTRA_TICKS} tick, stop first in the same "
      f"candle, no trades across pauses > {MAX_GAP_MINUTES} min, tick value {TICK_VALUE_USD} USD")
display(test_df.round(2))

Rules: costs 2.0 ticks, target fill +1 tick, stop first in the same candle, no trades across pauses > 45 min, tick value 5.0 USD


,Setup,Threshold,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,10bars,0.55,Model,2772,697.0,2.23,45.48,37.45,41.46,21.09,-5.21,-4.70,-5.63,0.90,-1.32,5/13,-18140.0,30810.0
1,10bars,0.55,Random direction (Ø 5),2772,698.6,2.23,49.78,38.28,40.83,20.90,-4.82,-4.24,-5.38,0.91,-1.23,8/13,-16809.0,26829.0
2,10bars,0.55,Always long,2772,698.0,2.23,100.00,39.97,39.11,20.92,-2.34,-2.34,NaN,0.95,-0.60,6/13,-8160.0,20600.0
3,30bars,0.38,Model,107111,7481.0,23.90,54.62,34.83,33.67,31.49,1.11,0.87,1.40,1.01,0.54,8/13,41525.0,82670.0
4,30bars,0.38,Random direction (Ø 5),107111,7485.0,23.91,49.51,34.06,34.56,31.38,-2.64,-2.69,-2.59,0.97,-1.28,2/13,-98792.0,157680.0
5,30bars,0.38,Always long,107111,7478.0,23.89,100.00,34.37,34.17,31.47,-2.21,-2.21,NaN,0.97,-1.08,4/13,-82770.0,96460.0


### 8.1 For Information Only: All Thresholds on the Test Data

Do **not** pick the threshold from this table – that would be fitting to the test year. It only shows whether the
result is stable around the chosen threshold.

In [12]:
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(outcomes_test[name], te[mask], direction[mask])
        rows.append({"Threshold": thr, **summarize(tr, df_test["timestamp"], test_days)})
    print(f"[{name}] test results per threshold (chosen on validation: {chosen[name]})")
    display(pd.DataFrame(rows).round(3))

[10bars] test results per threshold (chosen on validation: 0.55)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,26882,85.885,49.728,33.692,33.941,32.367,-2.278,-1.669,-2.881,0.954,-3.579,2/13,-306185.0,317410.0
1,0.325,24632,78.696,50.032,34.626,34.415,30.960,-1.547,-0.985,-2.110,0.969,-2.334,4/13,-190555.0,213720.0
2,0.350,21223,67.805,50.342,35.537,34.934,29.529,-1.408,-0.930,-1.893,0.971,-1.983,5/13,-149425.0,158200.0
3,0.375,16636,53.150,50.589,36.295,35.748,27.957,-1.473,-0.405,-2.566,0.970,-1.842,5/13,-122510.0,154785.0
4,0.400,11609,37.089,51.426,36.334,36.351,27.315,-2.293,-1.170,-3.483,0.954,-2.409,3/13,-133125.0,145905.0
5,0.425,7405,23.658,51.087,36.111,37.718,26.172,-3.842,-2.608,-5.132,0.924,-3.225,3/13,-142265.0,156145.0
6,0.450,4518,14.435,49.956,37.362,38.004,24.635,-2.470,0.498,-5.433,0.951,-1.610,5/13,-55795.0,71770.0
7,0.475,2769,8.847,49.332,37.270,38.678,24.052,-3.342,-1.806,-4.837,0.934,-1.700,5/13,-46265.0,53060.0
8,0.500,1696,5.419,47.524,36.380,39.800,23.821,-5.021,-4.809,-5.213,0.902,-2.008,2/13,-42580.0,51050.0
9,0.525,1115,3.562,46.906,37.489,40.359,22.152,-3.637,-1.916,-5.157,0.929,-1.169,5/13,-20275.0,34040.0


[30bars] test results per threshold (chosen on validation: 0.375)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,9602,30.677,53.249,34.243,32.785,32.972,0.674,0.617,0.739,1.008,0.368,7/13,32370.0,91095.0
1,0.325,9165,29.281,53.530,34.392,33.104,32.504,0.683,-0.461,2.001,1.008,0.365,5/13,31300.0,89505.0
2,0.350,8434,26.946,54.185,34.657,33.258,32.084,0.859,-0.315,2.248,1.010,0.443,5/13,36245.0,70775.0
3,0.375,7481,23.901,54.618,34.835,33.672,31.493,1.110,0.870,1.399,1.014,0.540,8/13,41525.0,82670.0
4,0.400,6381,20.387,55.101,35.089,34.697,30.215,-1.993,-2.971,-0.792,0.976,-0.897,5/13,-63575.0,109665.0
5,0.425,5185,16.565,56.104,35.911,34.773,29.315,-0.889,-1.360,-0.286,0.989,-0.361,6/13,-23035.0,71010.0
6,0.450,4023,12.853,56.550,36.266,34.004,29.729,1.946,0.599,3.700,1.024,0.700,9/13,39145.0,48015.0
7,0.475,3010,9.617,57.575,34.751,35.880,29.369,-4.131,-4.564,-3.543,0.951,-1.283,8/13,-62170.0,102890.0
8,0.500,2196,7.016,58.015,35.155,35.792,29.053,-3.080,-1.121,-5.786,0.963,-0.819,7/13,-33815.0,73335.0
9,0.525,1563,4.994,58.669,36.660,36.340,26.999,-1.381,-2.324,-0.042,0.984,-0.308,5/13,-10790.0,49420.0


## 8.2 Direction Strategy: Direction Model + Timing

* **Direction:** short if `P(up first)` is in the lowest `X %`, long if it is in the highest `X %` (cut-offs from the
  validation distribution, `DIRECTION_QUANTILES`).
* **Timing:** additionally the win model of the traded side must be at least at its validation quantile
  `TIMING_QUANTILES` (0 = no filter) – i.e. a move large enough to reach the target is expected.
* **Sides:** both, short only, long only (`DIRECTION_SIDES`).
* Same exits as above (volatility target / stop, stop first in the same candle, +1 tick for the target, time exit),
  one position at a time, costs included.

All combinations are evaluated on the **validation period**; the best t-value (with at least `MIN_VALIDATION_TRADES`
trades) is applied **once** to the test data and compared with random direction, always long and always short on exactly
the same entry candles. Long and short are reported separately.

In [13]:
def direction_signals(p_up, p_long, p_short, cfg):
    long_ = (p_up >= cfg["p_up_high"]) & (p_long >= cfg["p_long_min"])
    short_ = (p_up <= cfg["p_up_low"]) & (p_short >= cfg["p_short_min"])
    if cfg["side"] == "short":
        long_[:] = False
    elif cfg["side"] == "long":
        short_[:] = False
    return long_ | short_, np.where(long_, 1, -1)


dir_chosen, dir_test_rows, dir_trades, dir_signal_rows = {}, [], {}, {}
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    pu_va = direction_models[name].predict_proba(X_train_all.iloc[va])[:, 1]
    pu_te = direction_models[name].predict_proba(X_test.iloc[te])[:, 1]
    pl_va, ps_va = probs["val"][name][1], probs["val"][name][2]
    pl_te, ps_te = probs["test"][name][1], probs["test"][name][2]
    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()

    # ---- grid on validation
    grid = []
    for q in DIRECTION_QUANTILES:
        for tq in TIMING_QUANTILES:
            for side in DIRECTION_SIDES:
                cfg = {
                    "direction_quantile": q, "timing_quantile": tq, "side": side,
                    "p_up_low": float(np.quantile(pu_va, q)), "p_up_high": float(np.quantile(pu_va, 1 - q)),
                    "p_long_min": float(np.quantile(pl_va, tq)) if tq > 0 else -np.inf,
                    "p_short_min": float(np.quantile(ps_va, tq)) if tq > 0 else -np.inf,
                }
                mask, d = direction_signals(pu_va, pl_va, ps_va, cfg)
                tr = simulate(o_tr, va[mask], d[mask])
                grid.append({**cfg, **summarize(tr, df_train["timestamp"], val_days)})
    grid_df = pd.DataFrame(grid)
    eligible = grid_df[(grid_df["Trades"] >= MIN_VALIDATION_TRADES) & grid_df["t-value"].notna()]
    print(f"[{name}] validation: all combinations")
    display(grid_df.drop(columns=["p_long_min", "p_short_min"]).round(3))
    if eligible.empty:
        print(f"[{name}] no combination with ≥ {MIN_VALIDATION_TRADES} validation trades")
        continue
    best = eligible.loc[eligible["t-value"].idxmax()]
    cfg = {k: best[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
    dir_chosen[name] = cfg
    print(f"[{name}] chosen on validation: side {cfg['side']}, direction {cfg['direction_quantile']:.0%}, "
          f"timing quantile {cfg['timing_quantile']} -> validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
          f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")

    # ---- test, applied once
    mask, d = direction_signals(pu_te, pl_te, ps_te, cfg)
    rows_sig = te[mask]
    dir_signal_rows[name] = (rows_sig, d[mask])
    n_sig = int(mask.sum())
    strategies = {
        "Direction model": [d[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=n_sig) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(n_sig, dtype=int)],
        "Always short": [-np.ones(n_sig, dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for dd in dirs:
            tr = simulate(o_te, rows_sig, dd)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Direction model":
                dir_trades[name] = tr
        row = {"Setup": name, "Side": cfg["side"], "Direction quantile": cfg["direction_quantile"],
               "Timing quantile": cfg["timing_quantile"], "Strategy": strategy, "Signal candles": n_sig}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        dir_test_rows.append(row)

    # ---- for information only: all combinations on the test data
    info = []
    for _, g in grid_df.iterrows():
        c = {k: g[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
        m2, d2 = direction_signals(pu_te, pl_te, ps_te, c)
        info.append({"direction_quantile": c["direction_quantile"], "timing_quantile": c["timing_quantile"],
                     "side": c["side"], **summarize(simulate(o_te, te[m2], d2[m2]), df_test["timestamp"], test_days)})
    dir_signal_rows[name + "_info"] = pd.DataFrame(info)

print("TEST – direction strategy with the combination chosen on validation:")
dir_test_df = pd.DataFrame(dir_test_rows)
display(dir_test_df.round(2))
for s in SETUPS:
    if s["name"] + "_info" in dir_signal_rows:
        print(f"[{s['name']}] for information only – all combinations on the test data (do not pick from here):")
        display(dir_signal_rows[s["name"] + "_info"].round(3))

[10bars] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.386,0.624,1908,11.925,46.436,34.224,31.184,34.591,1.168,-4.453,6.041,1.024,0.488,3/7,11145.0,34665.0
1,0.05,0.00,short,0.386,0.624,1022,6.388,0.000,33.268,29.159,37.573,6.041,NaN,6.041,1.134,1.860,4/7,30870.0,17415.0
2,0.05,0.00,long,0.386,0.624,886,5.538,100.000,35.327,33.521,31.151,-4.453,-4.453,NaN,0.913,-1.262,1/7,-19725.0,36550.0
3,0.05,0.50,both,0.386,0.624,1609,10.056,48.291,36.420,32.940,30.640,1.401,-3.063,5.571,1.029,0.543,3/7,11275.0,30100.0
4,0.05,0.50,short,0.386,0.624,832,5.200,0.000,35.817,31.130,33.053,5.571,NaN,5.571,1.124,1.564,3/7,23175.0,16445.0
5,0.05,0.50,long,0.386,0.624,777,4.856,100.000,37.066,34.878,28.057,-3.063,-3.063,NaN,0.939,-0.820,2/7,-11900.0,27635.0
6,0.05,0.75,both,0.386,0.624,1335,8.344,51.086,37.753,33.558,28.689,2.780,-2.843,8.652,1.059,0.988,3/7,18555.0,26460.0
7,0.05,0.75,short,0.386,0.624,653,4.081,0.000,38.132,31.700,30.168,8.652,NaN,8.652,1.199,2.172,4/7,28250.0,14015.0
8,0.05,0.75,long,0.386,0.624,682,4.262,100.000,37.390,35.337,27.273,-2.843,-2.843,NaN,0.943,-0.718,2/7,-9695.0,20485.0
9,0.10,0.00,both,0.414,0.590,3514,21.962,47.553,32.555,32.242,35.202,-2.274,-3.668,-1.010,0.954,-1.296,2/7,-39955.0,83165.0


[10bars] chosen on validation: side short, direction 5%, timing quantile 0.75 -> validation Ø net +8.65 ticks, t-value 2.17, 653 trades
[30bars] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.313,0.707,763,5.087,47.575,32.765,33.421,33.814,-5.779,2.320,-13.128,0.932,-0.896,3/7,-22045.0,61715.0
1,0.05,0.00,short,0.313,0.707,400,2.667,0.000,29.000,33.000,38.000,-13.240,NaN,-13.240,0.847,-1.524,2/7,-26480.0,46415.0
2,0.05,0.00,long,0.313,0.707,363,2.420,100.000,36.915,34.160,28.926,2.320,2.320,NaN,1.028,0.242,3/7,4210.0,23565.0
3,0.05,0.50,both,0.313,0.707,704,4.693,48.580,33.523,33.949,32.528,-5.268,2.287,-12.406,0.937,-0.796,3/7,-18545.0,57770.0
4,0.05,0.50,short,0.313,0.707,362,2.413,0.000,29.834,33.149,37.017,-12.530,NaN,-12.530,0.854,-1.389,2/7,-22680.0,43345.0
5,0.05,0.50,long,0.313,0.707,342,2.280,100.000,37.427,35.088,27.485,2.287,2.287,NaN,1.028,0.235,3/7,3910.0,20085.0
6,0.05,0.75,both,0.313,0.707,604,4.027,49.007,33.940,36.258,29.801,-9.497,-4.459,-14.338,0.890,-1.327,3/7,-28680.0,57525.0
7,0.05,0.75,short,0.313,0.707,308,2.053,0.000,31.818,35.065,33.117,-14.338,NaN,-14.338,0.836,-1.454,2/7,-22080.0,39715.0
8,0.05,0.75,long,0.313,0.707,296,1.973,100.000,36.149,37.500,26.351,-4.459,-4.459,NaN,0.948,-0.429,3/7,-6600.0,22250.0
9,0.10,0.00,both,0.356,0.660,1362,9.080,47.797,33.333,34.141,32.526,-5.932,-5.447,-6.376,0.931,-1.217,3/7,-40395.0,89055.0


[30bars] chosen on validation: side long, direction 5%, timing quantile 0.0 -> validation Ø net +2.32 ticks, t-value 0.24, 363 trades
TEST – direction strategy with the combination chosen on validation:


,Setup,Side,Direction quantile,Timing quantile,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,10bars,short,0.05,0.75,Direction model,8525,2126.0,6.79,0.00,35.94,35.79,28.27,-0.74,NaN,-0.74,0.99,-0.32,7/13,-7850.0,35810.0
1,10bars,short,0.05,0.75,Random direction (Ø 5),8525,2128.6,6.80,49.95,35.53,36.19,28.28,-2.86,-4.39,-1.33,0.94,-1.25,5/13,-30403.0,42910.0
2,10bars,short,0.05,0.75,Always long,8525,2128.0,6.80,100.00,35.24,36.61,28.15,-3.92,-3.92,NaN,0.92,-1.72,4/13,-41695.0,65860.0
3,10bars,short,0.05,0.75,Always short,8525,2126.0,6.79,0.00,35.94,35.79,28.27,-0.74,NaN,-0.74,0.99,-0.32,7/13,-7850.0,35810.0
4,30bars,long,0.05,0.00,Direction model,13563,1085.0,3.47,100.00,32.63,32.44,34.93,-1.23,-1.23,NaN,0.99,-0.22,6/13,-6685.0,24400.0
5,30bars,long,0.05,0.00,Random direction (Ø 5),13563,1082.8,3.46,49.70,32.60,32.19,35.21,-2.08,-1.17,-2.97,0.98,-0.38,6/13,-11260.0,40460.0
6,30bars,long,0.05,0.00,Always long,13563,1085.0,3.47,100.00,32.63,32.44,34.93,-1.23,-1.23,NaN,0.99,-0.22,6/13,-6685.0,24400.0
7,30bars,long,0.05,0.00,Always short,13563,1082.0,3.46,0.00,32.16,32.53,35.30,-2.90,NaN,-2.90,0.97,-0.53,5/13,-15690.0,37015.0


[10bars] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,6207,19.831,54.084,33.076,33.382,33.543,-2.778,-3.345,-2.109,0.946,-2.047,5/13,-86200.0,107620.0
1,0.05,0.00,short,2851,9.109,0.000,33.076,33.287,33.637,-2.092,NaN,-2.092,0.959,-1.046,7/13,-29825.0,57260.0
2,0.05,0.00,long,3357,10.725,100.000,33.065,33.452,33.482,-3.345,-3.345,NaN,0.935,-1.812,5/13,-56140.0,62960.0
3,0.05,0.50,both,5315,16.981,53.641,34.506,34.468,31.025,-1.728,-1.289,-2.235,0.966,-1.193,5/13,-45910.0,77755.0
4,0.05,0.50,short,2464,7.872,0.000,34.456,34.862,30.682,-2.256,NaN,-2.256,0.956,-1.059,7/13,-27790.0,45930.0
5,0.05,0.50,long,2851,9.109,100.000,34.549,34.128,31.322,-1.289,-1.289,NaN,0.974,-0.653,8/13,-18375.0,35050.0
6,0.05,0.75,both,4564,14.581,53.418,35.824,35.123,29.053,-0.614,-0.506,-0.738,0.988,-0.394,5/13,-14015.0,55050.0
7,0.05,0.75,short,2126,6.792,0.000,35.936,35.795,28.269,-0.738,NaN,-0.738,0.985,-0.323,7/13,-7850.0,35810.0
8,0.05,0.75,long,2438,7.789,100.000,35.726,34.537,29.737,-0.506,-0.506,NaN,0.990,-0.237,7/13,-6165.0,24920.0
9,0.10,0.00,both,10765,34.393,54.594,33.284,33.135,33.581,-1.838,-2.706,-0.794,0.963,-1.789,5/13,-98910.0,116405.0


[30bars] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,2163,6.911,50.162,31.808,33.195,34.998,-4.037,-1.334,-6.759,0.953,-1.040,5/13,-43665.0,54705.0
1,0.05,0.00,short,1080,3.450,0.000,30.926,33.981,35.093,-6.779,NaN,-6.779,0.921,-1.241,5/13,-36605.0,43745.0
2,0.05,0.00,long,1085,3.466,100.000,32.627,32.442,34.931,-1.232,-1.232,NaN,0.985,-0.223,6/13,-6685.0,24400.0
3,0.05,0.50,both,1986,6.345,50.806,32.679,34.089,33.233,-4.278,-2.050,-6.580,0.950,-1.060,5/13,-42485.0,52735.0
4,0.05,0.50,short,978,3.125,0.000,32.106,34.969,32.924,-6.363,NaN,-6.363,0.926,-1.108,4/13,-31115.0,40655.0
5,0.05,0.50,long,1009,3.224,100.000,33.300,33.201,33.499,-2.071,-2.071,NaN,0.976,-0.365,5/13,-10450.0,25825.0
6,0.05,0.75,both,1728,5.521,52.315,32.639,34.317,33.044,-5.255,-5.052,-5.478,0.939,-1.222,5/13,-45405.0,48680.0
7,0.05,0.75,short,825,2.636,0.000,32.727,35.273,32.000,-5.222,NaN,-5.222,0.938,-0.845,5/13,-21540.0,31535.0
8,0.05,0.75,long,905,2.891,100.000,32.597,33.481,33.923,-5.396,-5.396,NaN,0.938,-0.901,4/13,-24415.0,27815.0
9,0.10,0.00,both,3735,11.933,50.094,33.360,32.691,33.949,-1.463,-1.538,-1.387,0.983,-0.490,8/13,-27315.0,83605.0


## 9. Write Test CSV with Signals, Save Models and Log

In [14]:
# Output = test period rows of the original file (all columns as text, timestamp unchanged) + probabilities and signals
test_text = pd.concat(
    [texts[f].iloc[[r]] for f, r in zip(df_test["_file"], df_test["_row"])] if len(texts) > 1
    else [texts[df_test["_file"].iloc[0]].iloc[df_test["_row"].to_numpy()]],
    ignore_index=True,
)
add = pd.DataFrame(index=df_test.index)
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    col_l, col_s, col_sig = f"P_long_{name}", f"P_short_{name}", f"Signal_{name}"
    add[col_l] = np.nan
    add[col_s] = np.nan
    add.loc[te, col_l] = np.round(p_l, 4)
    add.loc[te, col_s] = np.round(p_s, 4)
    add[col_sig] = 0
    if chosen[name] is not None:
        mask, direction = signals(p_l, p_s, chosen[name])
        add.loc[te[mask], col_sig] = direction[mask]
    add[f"Target_ticks_{name}"] = outcomes_test[name]["target_ticks"]
    add[f"Stop_ticks_{name}"] = outcomes_test[name]["stop_ticks"]
    add[f"P_up_first_{name}"] = np.nan
    add.loc[te, f"P_up_first_{name}"] = np.round(direction_models[name].predict_proba(X_test.iloc[te])[:, 1], 4)
    add[f"Trade_{name}"] = 0                         # trades actually taken (one position at a time)
    if name in test_trades and not test_trades[name].empty:
        add.loc[test_trades[name]["t"].to_numpy(), f"Trade_{name}"] = test_trades[name]["direction"].to_numpy()
    add[f"DirSignal_{name}"] = 0                     # direction strategy (section 8.2): signal / trades taken
    add[f"DirTrade_{name}"] = 0
    if name in dir_signal_rows:
        rows_sig, dirs = dir_signal_rows[name]
        add.loc[rows_sig, f"DirSignal_{name}"] = dirs
    if name in dir_trades and not dir_trades[name].empty:
        add.loc[dir_trades[name]["t"].to_numpy(), f"DirTrade_{name}"] = dir_trades[name]["direction"].to_numpy()

out = pd.concat([test_text.reset_index(drop=True), add.reset_index(drop=True)], axis=1)
out.to_csv(OUTPUT_CSV_PATH, sep=";", index=False, encoding="utf-8-sig")
print(f"✓ Written: {OUTPUT_CSV_PATH} ({len(out):,} test rows, new columns: {list(add.columns)})")

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_rows = []
for s in SETUPS:
    name = s["name"]
    path = MODEL_DIR / f"barrier_{SYMBOL}_{name}_{run_ts}.joblib"
    joblib.dump({
        "models": models[name], "direction_model": direction_models.get(name),
        "direction_strategy": dir_chosen.get(name),
        "features": FEATURE_COLUMNS, "setup": s, "threshold": chosen[name],
        "vol_lookback": VOL_LOOKBACK, "min_barrier_ticks": MIN_BARRIER_TICKS,
        "trained_until": str(df_train["timestamp"].iloc[val_start - 1]), "test_period_start": str(test_start), "input_columns": INPUT_COLUMNS,
        "use_time_features": USE_TIME_FEATURES, "tick_size": TICK_SIZE,
        "rsi_periods": RSI_PERIODS, "twap_windows": TWAP_WINDOWS, "vwap_windows": VWAP_WINDOWS,
        "use_session_features": USE_SESSION_FEATURES, "session_start_hour_et": SESSION_START_HOUR_ET, "us_open_et": US_OPEN_ET, "opening_range_minutes": OPENING_RANGE_MINUTES,
    }, path)
    joblib.dump(joblib.load(path), MODEL_DIR / f"barrier_{SYMBOL}_{name}_latest.joblib")
    print(f"✓ Saved: {path.resolve()}")
    model_row = test_df[(test_df["Setup"] == name) & (test_df["Strategy"] == "Model")]
    auc = pd.DataFrame(auc_rows)
    log_rows.append({
        "run_timestamp": run_ts, "setup": name, "horizon": s["horizon"], "mode": s.get("mode", "fixed"),
        "target": s.get("target_ticks", s.get("target_sigma")), "stop": s.get("stop_ticks", s.get("stop_sigma")),
        "threshold": chosen[name],
        "auc_direction_val": next((r["AUC direction validation"] for r in direction_rows if r["Setup"] == name), None),
        "auc_direction_test": next((r["AUC direction test"] for r in direction_rows if r["Setup"] == name), None),
        "auc_val_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC validation"].iloc[0]),
        "auc_val_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC validation"].iloc[0]),
        "auc_test_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC test"].iloc[0]),
        "auc_test_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC test"].iloc[0]),
        **({k: model_row.iloc[0][k] for k in ["Trades", "Ø net (ticks)", "Profit factor", "t-value",
                                             "Months in profit", "Net total USD/contract", "Max drawdown USD"]}
           if len(model_row) else {}),
        **({f"dir_{k}": dir_chosen[name][k] for k in ["side", "direction_quantile", "timing_quantile"]}
           if name in dir_chosen else {}),
        **({f"dir_{k}": v for k, v in dir_test_df[(dir_test_df["Setup"] == name)
                                                    & (dir_test_df["Strategy"] == "Direction model")].iloc[0][
               ["Trades", "Ø net (ticks)", "Profit factor", "t-value", "Months in profit", "Net total USD/contract"]].items()}
           if len(dir_test_df) and ((dir_test_df["Setup"] == name) & (dir_test_df["Strategy"] == "Direction model")).any() else {}),
        "costs_ticks": COST_PER_TRADE_TICKS, "train_row_stride": TRAIN_ROW_STRIDE, "model_path": str(path.resolve()),
    })
log_df = pd.DataFrame(log_rows)
if LOG_PATH.exists():
    log_df = pd.concat([pd.read_csv(LOG_PATH, sep=";"), log_df], ignore_index=True)
log_df.to_csv(LOG_PATH, sep=";", index=False)
print(f"✓ Logged to {LOG_PATH.resolve()}")

✓ Written: E:\Programierung\Daten\NQXV24\Candelexport_NQ_24_RangeXV_08.08.2021_13.03.2026_test_signals.csv (224,104 test rows, new columns: ['P_long_10bars', 'P_short_10bars', 'Signal_10bars', 'Target_ticks_10bars', 'Stop_ticks_10bars', 'P_up_first_10bars', 'Trade_10bars', 'DirSignal_10bars', 'DirTrade_10bars', 'P_long_30bars', 'P_short_30bars', 'Signal_30bars', 'Target_ticks_30bars', 'Stop_ticks_30bars', 'P_up_first_30bars', 'Trade_30bars', 'DirSignal_30bars', 'DirTrade_30bars'])
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_NQ_XV24\barrier_NQ_10bars_20261027_131403.joblib
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_NQ_XV24\barrier_NQ_30bars_20261027_131403.joblib
✓ Logged to E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_NQ_XV24\barrier_log.csv
